# 07 · Chronos-2 fleet model for production

One pooled Chronos-2, LoRA fine-tuned on every M1 meter of the raw file (each device and target
is one series), with the calendar and the site weather as covariates, exactly as the forecast
server builds them. Hourly data follow demo3's `silver_to_forecast_ready`. Training labels stop
before TRAIN_END (None: all data); the merged weights and `model_card.json` land in
`runs/nb07/<RUN_NAME>/model/`, ready for the server. When data exist after TRAIN_END, a short
replay compares zero-shot and fine-tuned. It needs a GPU, so the last cell runs it on the GX10.

In [ ]:
# What to train on
TARGETS = ["grid_export", "grid_import"]   # grid_export only on devices with PV (see PV_MIN_KWH)
DEVICES = None                             # None: every M1 device in the raw file; or a list of ids
TRAIN_END = None                           # local date (Europe/Rome), labels before it; None: all
PV_MIN_KWH = 50                            # PV: more kWh than this exported before TRAIN_END

# Chronos-2 LoRA fine-tune (r=8, alpha=16), same call as notebook 05
MODEL_ID = "amazon/chronos-2"
STEPS = 1000
LR = 1e-4
CONTEXT = 2048                             # hours per training window and per forecast
HORIZON = 168                              # training prediction length, the server max horizon
QUANTILES = [0.1, 0.25, 0.5, 0.75, 0.9]
FT_BATCH_SIZE = 64                         # series per training step (8 per item: 1 + 7 covariates)
PRED_BATCH_SIZE = 256
GPU_MEMORY_FRACTION = 0.12                 # of the GX10 ~121 GB unified memory (Qwen holds ~65)
SEED = 42

# Sanity replay after TRAIN_END (skipped when TRAIN_END is None)
EVAL_DAYS = 14                             # daily origins at local midnight from TRAIN_END
EVAL_HORIZON = 48

# Output
RUN_NAME = None                            # None: "fleet-<timestamp>"
LOG_MLFLOW = True                          # MLFLOW_TRACKING_URI if set, else runs/mlflow.db
MLFLOW_EXPERIMENT = "nb07-chronos2-fleet"

In [ ]:
import json
import os
import time
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
from dateutil.easter import easter

PARAMS = {k: v for k, v in dict(globals()).items() if k.isupper()}
NB_DIR = Path.cwd().resolve()
if not (NB_DIR / "data").exists() and (NB_DIR / "notebooks" / "data").exists():
    NB_DIR = NB_DIR / "notebooks"
REPO_ROOT = next(p for p in [NB_DIR, *NB_DIR.parents] if (p / "pyproject.toml").exists())
DATA_DIR = NB_DIR / "data"
RUN_NAME = RUN_NAME or f"fleet-{time.strftime('%Y%m%d-%H%M')}"
OUT_DIR = REPO_ROOT / "runs" / "nb07" / RUN_NAME
MODEL_DIR = OUT_DIR / "model"
LOCAL_TZ = "Europe/Rome"
T_END = None if TRAIN_END is None else pd.Timestamp(TRAIN_END, tz=LOCAL_TZ).tz_convert("UTC")

# demo3 silver_to_forecast_ready: the 15-min kWh of an hour are summed, an hour with n < 4
# quarters is scaled by 4/n, then one gapless hourly grid, 1-hour gaps interpolated, clip at 0.
raw = pd.read_parquet(DATA_DIR / "raw_meters_m1.parquet")
raw = raw[raw["meter_type"].eq("M1") & (raw["device_id"].isin(DEVICES) if DEVICES else True)]
raw["ts_hour"] = pd.to_datetime(raw["ts"], utc=True).dt.floor("h")
hourly = raw.groupby(["ts_hour", "device_id"]).agg(
    grid_import=("consumption_kwh", "sum"), grid_export=("production_kwh", "sum"),
    n=("consumption_kwh", "size"))
scale = (4 / hourly["n"]).clip(lower=1)
hourly = hourly[["grid_export", "grid_import"]].mul(scale, axis=0).clip(lower=0)
FULL = pd.date_range(*hourly.index.get_level_values("ts_hour")[[0, -1]], freq="h")
WIDE = {t: hourly[t].unstack("device_id").reindex(FULL).interpolate(method="linear", limit=1)
        .clip(lower=0) for t in ["grid_export", "grid_import"]}
ALL_DEVICES = sorted(WIDE["grid_import"].columns)
train = FULL < T_END if T_END is not None else np.ones(len(FULL), bool)
EXPORT_KWH = WIDE["grid_export"][train].sum()
HAS_PV = (EXPORT_KWH > PV_MIN_KWH).to_dict()

WEATHER_COLS = ["global_tilted_irradiance", "temperature_2m", "cloud_cover"]
W = (pd.read_parquet(DATA_DIR / "processed_hourly.parquet", columns=["ts_hour", *WEATHER_COLS])
     .groupby("ts_hour")[WEATHER_COLS].first())
span = FULL[train & WIDE["grid_import"].notna().any(axis=1).to_numpy()]
span = pd.date_range(span[0], span[-1], freq="h")
WEATHER_COVERAGE = float(span.isin(W.dropna().index).mean())
print(f"run {RUN_NAME} | {len(ALL_DEVICES)} devices | hours {FULL[0]} .. {FULL[-1]}"
      f" | train end {T_END} | output {OUT_DIR}")
print(f"PV (> {PV_MIN_KWH} kWh exported): {sum(HAS_PV.values())} devices"
      f" {sorted(d for d, v in HAS_PV.items() if v)}")
print(f"weather {W.index[0]} .. {W.index[-1]} covers {WEATHER_COVERAGE:.2%} of the training"
      f" hours {span[0]} .. {span[-1]}")
assert WEATHER_COVERAGE > 0.95, "the site weather does not cover the training span"

In [ ]:
# Series and covariates on one hourly UTC grid (a context is an array slice). Covariates as the
# server builds them: local hour and day of week as sin/cos, the Italian national holidays, and
# the site weather (gaps interpolated).
import gc

import torch
from chronos import Chronos2Pipeline
from transformers import set_seed

HOURS = pd.date_range(FULL[0] - pd.Timedelta(hours=CONTEXT), FULL[-1] + pd.Timedelta(hours=HORIZON),
                      freq="h")
local = HOURS.tz_convert(LOCAL_TZ)
years = range(local.year.min(), local.year.max() + 1)
HOLIDAYS = {pd.Timestamp(f"{y}-{md}").date() for y in years for md in [
    "01-01", "01-06", "04-25", "05-01", "06-02", "08-15", "11-01", "12-08", "12-25", "12-26"]}
HOLIDAYS |= {easter(y) + timedelta(days=1) for y in years}  # Easter Monday
COV = pd.DataFrame({"hour_sin": np.sin(2 * np.pi * local.hour / 24),
                    "hour_cos": np.cos(2 * np.pi * local.hour / 24),
                    "dow_sin": np.sin(2 * np.pi * local.dayofweek / 7),
                    "dow_cos": np.cos(2 * np.pi * local.dayofweek / 7),
                    "is_holiday": pd.Index(local.date).isin(HOLIDAYS).astype(float)}, index=HOURS)
COV[WEATHER_COLS] = W.reindex(HOURS).interpolate(limit_direction="both")
COV_ARR = COV.to_numpy(np.float32)
PAIRS = [(d, t) for t in TARGETS for d in ALL_DEVICES if t != "grid_export" or HAS_PV[d]]
Y = {(d, t): WIDE[t][d].reindex(HOURS).to_numpy(np.float32) for d, t in PAIRS}


def pos(ts):
    """Row of a UTC timestamp on HOURS."""
    return int((ts - HOURS[0]) // pd.Timedelta(hours=1))


def covariates(a, b):
    """The covariates of rows a..b-1 of HOURS, by name."""
    return dict(zip(COV.columns, COV_ARR[a:b].T, strict=True))


def forecast(pipe, keys, horizon):
    """Quantiles (series, lead, quantile) of each (device, target, origin) of `keys` from the
    CONTEXT hours before the origin, clipped at 0 and sorted across quantile levels."""
    inputs = [{"target": Y[d, t][p - CONTEXT:p], "past_covariates": covariates(p - CONTEXT, p),
               "future_covariates": covariates(p, p + horizon)}
              for d, t, p in ((d, t, pos(o)) for d, t, o in keys)]
    q, _ = pipe.predict_quantiles(inputs, prediction_length=horizon, quantile_levels=QUANTILES,
                                  batch_size=PRED_BATCH_SIZE, context_length=CONTEXT)
    return np.sort(np.maximum(0, torch.stack(q)[:, 0].float().cpu().numpy()), axis=-1)


# Training series: labels before TRAIN_END, from the first observation; Chronos needs at least
# 2 * HORIZON hours (min_past + prediction length), shorter series are left out.
P_END = pos(T_END) if T_END is not None else pos(FULL[-1]) + 1
TRAIN, SHORT = [], []
for d, t in PAIRS:
    y = Y[d, t][:P_END]
    start = int(np.argmax(np.isfinite(y)))
    if P_END - start < 2 * HORIZON or not np.isfinite(y).any():
        SHORT.append(f"{d}/{t}")
        continue
    TRAIN.append({"target": y[start:], "past_covariates": covariates(start, P_END),
                  "future_covariates": {c: None for c in COV.columns}})
print(f"{len(PAIRS)} series, {len(TRAIN)} trained on, too short: {SHORT}")

assert torch.cuda.is_available(), "Chronos-2 fine-tuning needs a GPU: run it with gx10_run()"
torch.cuda.set_per_process_memory_fraction(GPU_MEMORY_FRACTION)
PIPE = Chronos2Pipeline.from_pretrained(MODEL_ID, device_map="cuda")
print(f"{MODEL_ID} on {torch.cuda.get_device_name(0)} | covariates {list(COV.columns)}")

In [ ]:
# Fine-tune, merge the adapters, save the plain Chronos-2 weights and the model card, then check
# that the saved directory reloads to the same forecasts.
OUT_DIR.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
t0 = time.time()
ft = PIPE.fit(TRAIN, prediction_length=HORIZON, finetune_mode="lora", context_length=CONTEXT,
              learning_rate=float(LR), num_steps=int(STEPS), batch_size=FT_BATCH_SIZE,
              output_dir=OUT_DIR / "trainer", remove_printer_callback=True, seed=SEED,
              disable_tqdm=True)
TRAIN_SECONDS = time.time() - t0
lora = ft.model.peft_config["default"]
# fit() leaves the model in train mode (dropout on): eval() for deterministic forecasts
FT = Chronos2Pipeline(model=ft.model.merge_and_unload().eval())
FT.save_pretrained(MODEL_DIR)
CARD = {
    "model_name": "chronos2-fleet", "model_version": RUN_NAME, "base_model": MODEL_ID,
    "context_length": CONTEXT, "max_horizon": HORIZON, "quantiles": QUANTILES,
    "covariates": list(COV.columns), "weather_covariates": WEATHER_COLS, "local_tz": LOCAL_TZ,
    "train_end": None if T_END is None else T_END.tz_convert(LOCAL_TZ).isoformat(),
    "n_devices": len(ALL_DEVICES), "targets": TARGETS,
    "finetune": {"mode": "lora", "steps": int(STEPS), "lr": float(LR), "context": CONTEXT,
                 "lora_r": lora.r, "lora_alpha": lora.lora_alpha, "seed": SEED},
    "created_at": pd.Timestamp.now(tz="UTC").isoformat()}
(MODEL_DIR / "model_card.json").write_text(json.dumps(CARD, indent=2))

keys = [(d, t, HOURS[P_END]) for d, t in PAIRS
        if np.isfinite(Y[d, t][P_END - CONTEXT:P_END]).any()][:6]
RELOADED = Chronos2Pipeline.from_pretrained(MODEL_DIR, device_map="cuda")
np.testing.assert_allclose(forecast(RELOADED, keys, HORIZON), forecast(FT, keys, HORIZON),
                           rtol=1e-3, atol=1e-3)
del ft, RELOADED
gc.collect()
torch.cuda.empty_cache()
PEAK_GIB = torch.cuda.max_memory_allocated() / 2**30
(OUT_DIR / "run.json").write_text(json.dumps({
    "train_seconds": round(TRAIN_SECONDS), "peak_gpu_gib": round(PEAK_GIB, 2),
    "series_trained": len(TRAIN), "too_short": SHORT, "weather_coverage": WEATHER_COVERAGE,
    "export_kwh": EXPORT_KWH.round(1).to_dict(), "has_pv": HAS_PV}, indent=2))
print(f"fine-tuned {len(TRAIN)} series in {TRAIN_SECONDS:.0f}s | peak GPU {PEAK_GIB:.1f} GiB"
      f" | reload check ok on {len(keys)} series | {sorted(p.name for p in MODEL_DIR.iterdir())}")

In [ ]:
# Sanity replay: daily origins after TRAIN_END (never before it, those labels were trained on),
# EVAL_HORIZON-hour paths, MAE in kWh/h per target; naive168 repeats the value of a week earlier.
EVAL = None
if T_END is not None:
    origins = pd.date_range(T_END.tz_convert(LOCAL_TZ), periods=EVAL_DAYS, freq="D")
    origins = origins.tz_convert("UTC")
    origins = origins[origins + pd.Timedelta(hours=EVAL_HORIZON - 1) <= FULL[-1]]
    keys = [(d, t, o) for o in origins for d, t in PAIRS
            if np.isfinite(Y[d, t][pos(o) - CONTEXT:pos(o)]).sum() >= 24]
    if keys:
        lead = np.arange(EVAL_HORIZON)
        actual = np.stack([Y[d, t][pos(o) + lead] for d, t, o in keys])
        preds = {"zeroshot": forecast(PIPE, keys, EVAL_HORIZON)[..., QUANTILES.index(0.5)],
                 "finetuned": forecast(FT, keys, EVAL_HORIZON)[..., QUANTILES.index(0.5)],
                 "naive168": np.stack([Y[d, t][pos(o) + lead - 168] for d, t, o in keys])}
        target = np.array([t for _, t, _ in keys])
        EVAL = pd.DataFrame({m: {t: np.nanmean(np.abs(p - actual)[target == t]) for t in TARGETS}
                             for m, p in preds.items()})
        EVAL["series"] = pd.Series(target).value_counts()
        EVAL.to_csv(OUT_DIR / "eval.csv")
        print(f"{len(origins)} origins {origins[0]} .. {origins[-1]}, {len(keys)} paths")
        display(EVAL.round(4))

In [ ]:
if LOG_MLFLOW:
    import mlflow

    default_uri = f"sqlite:///{REPO_ROOT / 'runs' / 'mlflow.db'}"
    mlflow.set_tracking_uri(os.environ.get("MLFLOW_TRACKING_URI", default_uri))
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    with mlflow.start_run(run_name=RUN_NAME):
        mlflow.log_params({**PARAMS, "host": os.uname().nodename, "n_devices": len(ALL_DEVICES),
                           "n_series": len(TRAIN), "n_pv": sum(HAS_PV.values())})
        mlflow.log_metrics({"train_seconds": TRAIN_SECONDS, "peak_gpu_gib": PEAK_GIB,
                            "weather_coverage": WEATHER_COVERAGE})
        if EVAL is not None:
            mlflow.log_metrics({f"eval.{m}.mae.{t}": v for m in preds for t, v in EVAL[m].items()})
        for f in [MODEL_DIR / "model_card.json", OUT_DIR / "run.json", OUT_DIR / "eval.csv"]:
            if f.exists():
                mlflow.log_artifact(str(f))
    print("logged to", mlflow.get_tracking_uri())

In [ ]:
# Run this notebook on the GX10 (the only GPU): save the notebook, run the first two code cells
# and this one, then call gx10_run(); keyword arguments override the top cell, e.g.
# gx10_run(RUN_NAME="fleet-eval-20260901", TRAIN_END="2026-09-01").
# gx10_status() shows running jobs, gx10_pull() brings runs/nb07 (models included) and its
# MLflow runs back to runs/gx10/nb07. The job runs in the GPU sandbox container (NVIDIA vLLM
# image, venv /w/.venv-nv with chronos-forecasting, peft, papermill), which mounts
# ~/chronos-sandbox at /w and is capped at 24 GB; results land in ~/chronos-sandbox/runs/nb07.
import shlex
import subprocess

GX10_HOST = os.environ.get("GX10_HOST")                # user@host of the box, set it in your shell
GX10_DIR = "chronos-sandbox"                        # on the box, relative to the home folder
GX10_CONTAINER = "chronos-sandbox-nv"
GX10_MIN_FREE_GB = 15                               # the box freezes if memory runs out
NB_NAME = "07_chronos2_fleet_train"


def gx10_run(**overrides):
    """Copy this notebook and its data to a job folder on the box, then run it there in the GPU
    container, with `overrides` (parameter name: value) on top of the top cell."""
    if not GX10_HOST:
        raise RuntimeError("set GX10_HOST (user@host) in the environment first")
    free = subprocess.run(["ssh", GX10_HOST, "free -g | awk '/Mem:/ {print $7}'"], check=True,
                          capture_output=True, text=True).stdout
    assert int(free) >= GX10_MIN_FREE_GB, f"only {free.strip()} GB available on the GX10"
    busy = subprocess.run(["ssh", GX10_HOST, f"docker exec {GX10_CONTAINER} pgrep -f papermill"],
                          capture_output=True, text=True).stdout
    assert not busy.strip(), "a GPU job is already running in the sandbox, see gx10_status()"
    p = {**PARAMS, **overrides}
    p["RUN_NAME"] = p["RUN_NAME"] or f"fleet-{time.strftime('%Y%m%d-%H%M')}"
    job = p["RUN_NAME"]
    jobdir = f"runs/nb07/jobs/{job}"
    subprocess.run(["ssh", GX10_HOST, f"mkdir -p {GX10_DIR}/{jobdir}/data"], check=True)
    subprocess.run(["rsync", "-az", str(NB_DIR / f"{NB_NAME}.ipynb"),
                    f"{GX10_HOST}:{GX10_DIR}/{jobdir}/"], check=True)
    inputs = [str(DATA_DIR / f) for f in ["raw_meters_m1.parquet", "processed_hourly.parquet"]]
    subprocess.run(["rsync", "-az", *inputs, f"{GX10_HOST}:{GX10_DIR}/{jobdir}/data/"], check=True)
    w = f"/w/{jobdir}"
    remote = (
        # only docker goes to the background
        f"cd {GX10_DIR}/{jobdir} || exit 1; "
        f"nohup docker exec -e HOME=/tmp -e MLFLOW_TRACKING_URI=sqlite:////w/runs/nb07/mlflow.db "
        f"{GX10_CONTAINER} /w/.venv-nv/bin/papermill {w}/{NB_NAME}.ipynb "
        f"{w}/{NB_NAME}.out.ipynb --cwd {w} -y {shlex.quote(json.dumps(p))} "
        "> log.txt 2>&1 < /dev/null &")
    subprocess.run(["ssh", GX10_HOST, remote], check=True)
    print(f"started nb07 job {job} on {GX10_HOST}")


def gx10_status():
    """Running nb07 jobs with the last lines of their logs, and the box's free memory."""
    remote = (f"cd {GX10_DIR}/runs/nb07/jobs; "
              f"for j in $(docker exec {GX10_CONTAINER} pgrep -af papermill "
              "| grep -o 'jobs/[^/]*' | sort -u | cut -d/ -f2); do echo \"running: $j\"; "
              "tail -c 600 $j/log.txt | tr '\\r' '\\n' | tail -n 3; done; "
              "echo latest: $(ls -td */ | head -1); free -g | head -2")
    print(subprocess.run(["ssh", GX10_HOST, remote], capture_output=True, text=True).stdout)


def gx10_pull():
    """Copy runs/nb07 (models included, job inputs excluded) from the box to runs/gx10/nb07/
    here, then its MLflow runs to our tracking server."""
    local = REPO_ROOT / "runs" / "gx10" / "nb07"
    local.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-az", "--exclude", "jobs/*/data/",
                    f"{GX10_HOST}:{GX10_DIR}/runs/nb07/", str(local)], check=True)
    gx10_to_mlflow(local / "mlflow.db")


def gx10_to_mlflow(db):
    """Copy the box's runs (params, metrics, tags) to MLflow here, skipping those already copied."""
    from mlflow.entities import Metric, Param
    from mlflow.tracking import MlflowClient

    src = MlflowClient(f"sqlite:///{db}")
    uri = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{REPO_ROOT / 'runs' / 'mlflow.db'}")
    dst = MlflowClient(uri)
    copied = 0
    for exp in src.search_experiments():
        target = dst.get_experiment_by_name(exp.name)
        exp_id = target.experiment_id if target else dst.create_experiment(exp.name)
        # The box's run id travels as a tag, so pulling twice never copies a run twice.
        done = {r.data.tags.get("gx10.run_id")
                for r in dst.search_runs([exp_id], max_results=50000)}
        for run in src.search_runs([exp.experiment_id], max_results=50000):
            if run.info.run_id in done:
                continue
            tags = {**run.data.tags, "gx10.run_id": run.info.run_id, "gx10.host": GX10_HOST}
            new = dst.create_run(exp_id, start_time=run.info.start_time, tags=tags,
                                 run_name=run.info.run_name).info.run_id
            metrics = [Metric(m.key, m.value, m.timestamp, m.step) for k in run.data.metrics
                       for m in src.get_metric_history(run.info.run_id, k)]
            params = [Param(k, v) for k, v in run.data.params.items()]
            for i in range(0, len(metrics), 1000):  # log_batch caps: 1000 metrics, 100 params
                dst.log_batch(new, metrics=metrics[i:i + 1000])
            for i in range(0, len(params), 100):
                dst.log_batch(new, params=params[i:i + 100])
            dst.set_terminated(new, run.info.status, run.info.end_time)
            copied += 1
    print(f"copied {copied} new run(s) from {db} to {uri}")

In [ ]:
# gx10_run()
# gx10_status()
# gx10_pull()